# Lecture 6: Price Elasticity of Demand
## Estimating How Prices Affect Walmart Store Sales

---

### Learning Objectives

By completing this notebook you will be able to:

1. Explore and visualise the relationship between prices, macroeconomic factors, and sales
2. Build log-log regression models and interpret coefficients as elasticities
3. Perform backward variable selection using statistical significance
4. Diagnose regression problems: multicollinearity (VIF) and autocorrelation (Durbin-Watson)
5. Use store fixed effects to control for unobserved store heterogeneity
6. Interpret the business implications of elasticity estimates
7. Apply Ridge and Lasso regularisation to handle multicollinearity and compare their impact on elasticity estimates

### Dataset

We use the **Walmart Weekly Sales** dataset with 6,435 store-week observations across 45 stores:

| Variable | Description |
|----------|-------------|
| `Store` | Store number (1-45) |
| `Date` | Week of sales |
| `Weekly_Sales` | Total weekly sales revenue (\$) |
| `Holiday_Flag` | Whether the week includes a holiday (1 = Yes, 0 = No) |
| `Temperature` | Average temperature in the region (°F) |
| `Fuel_Price` | Cost of fuel in the region (\$/gallon) |
| `CPI` | Consumer Price Index — proxy for general price level |
| `Unemployment` | Regional unemployment rate (%) |

---
## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = '../data/Walmart.csv'

---
## 1. Load and Explore the Data

### Task 1.1: Load the dataset and inspect its structure

Load `Walmart.csv` into a DataFrame. Print its shape, column types, and summary statistics.

In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=['Date'], dayfirst=True)
print(data.shape); print(data.dtypes); data.describe()

### Task 1.2: Create log-transformed variables

Create two new columns: `log_CPI` and `log_Weekly_Sales` using the natural log.

**Why logs?** In a log-log model, the coefficient on `ln(CPI)` is directly interpretable as an elasticity:

$$\ln(\text{Weekly\_Sales}) = \beta_0 + \beta_1 \ln(\text{CPI}) + \ldots$$

Here $\beta_1$ represents: *a 1% increase in CPI is associated with a $\beta_1$% change in weekly sales*.

In [ ]:
data['log_CPI'], data['log_Weekly_Sales'] = np.log(data.CPI), np.log(data.Weekly_Sales)

### Task 1.3: Correlation matrix

Compute and visualise the correlation matrix between `log_Weekly_Sales`, `log_CPI`, `Holiday_Flag`, `Temperature`, `Fuel_Price`, and `Unemployment`.

**Hint:** Use `sns.heatmap()` with `annot=True` for a clear visualisation.

In [ ]:
cols = ['log_Weekly_Sales', 'log_CPI', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'Unemployment']
sns.heatmap(data[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0); plt.title('Correlation matrix'); plt.show()

### Task 1.4: Scatterplots of predictors vs log sales

Create scatterplots of each predictor (`log_CPI`, `Temperature`, `Fuel_Price`, `Unemployment`) against `log_Weekly_Sales`, coloured by `Holiday_Flag`.

**Hint:** Use a 2x2 subplot grid for a compact layout.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, c in zip(axes.flat, ['log_CPI', 'Temperature', 'Fuel_Price', 'Unemployment']): sns.scatterplot(data=data, x=c, y='log_Weekly_Sales', hue='Holiday_Flag', s=8, alpha=.5, ax=ax)
plt.tight_layout(); plt.show()

### Discussion

**Answer the following:**
- Which predictors show the strongest correlation with log weekly sales?
- Do you notice any patterns in the scatterplots (e.g., banding, clusters)? What might explain them?
- Are there any pairs of predictors that are highly correlated with each other? Why could this be a problem?

- **Strongest correlations with log sales:** all are weak; `Unemployment` (negative) and `log_CPI` (negative) are the largest, Temperature/Fuel_Price are near zero.
- **Patterns:** strong horizontal **banding** in log sales — each band is a store with its own baseline size. `log_CPI` also forms vertical **clusters** because CPI is regional (stores in the same region share CPI). Holiday weeks sit slightly higher.
- **Correlated predictors:** `Fuel_Price` and `log_CPI` (both trend upward over time). This causes **multicollinearity** → unstable, inflated standard errors and hard-to-interpret coefficients.

---
## 2. Baseline Log-Log Regression

### Task 2.1: Fit a full model with all predictors

Fit an OLS regression of `log_Weekly_Sales` on all five predictors: `Holiday_Flag`, `Temperature`, `Fuel_Price`, `Unemployment`, `log_CPI`.

**Hint:** Use `sm.OLS()` with `sm.add_constant()`. Print the full summary.

In [ ]:
y = data.log_Weekly_Sales
fit = lambda X: sm.OLS(y, sm.add_constant(data[X])).fit()
m_full = fit(['Holiday_Flag', 'Temperature', 'Fuel_Price', 'Unemployment', 'log_CPI']); print(m_full.summary())

### Discussion

**Answer the following:**
- What is the R-squared? Is it high or low? Why might this be the case?
- Which predictors are statistically significant (p < 0.05) and which are not?
- What is the coefficient on `log_CPI`? How do you interpret it as an elasticity?
- Look at the Durbin-Watson statistic. What value is it? What does it suggest?

- **R²** is very low (a few %): the model ignores the main driver of sales — store-level differences (size, location).
- **Significant:** `Unemployment`, `log_CPI`, `Temperature`; **not significant:** `Fuel_Price` (and `Holiday_Flag` borderline/weak).
- **log_CPI coefficient** is negative: a 1% rise in CPI is associated with a |β|% fall in weekly sales.
- **Durbin-Watson** is close to 0 → strong positive autocorrelation of residuals (consecutive weeks of the same store are very similar).

### Task 2.2: Backward selection — drop non-significant predictors

Iteratively remove the least significant predictor and re-fit the model until all remaining predictors are significant.

**Steps:**
1. Identify the predictor with the highest p-value above 0.05
2. Remove it and re-fit
3. Repeat until all predictors have p < 0.05

In [ ]:
X = ['Holiday_Flag', 'Temperature', 'Unemployment', 'log_CPI']
m2 = fit(X); print(m2.summary())

In [ ]:
# keep dropping the least significant predictor until all p < 0.05
while (p := fit(X).pvalues.drop('const')).max() > .05: print('Dropping', p.idxmax()); X.remove(p.idxmax())
model = fit(X); print(model.summary())

### Discussion

**Answer the following:**
- What is the elasticity of sales with respect to CPI in your final model?
- Is demand elastic or inelastic? What does this mean for Walmart's business?
- How does the unemployment coefficient translate into a business insight?

- **CPI elasticity:** the `log_CPI` coefficient of the final model (negative).
- **Inelastic** (|β| < 1): sales fall less than proportionally when prices rise — Walmart, as a low-cost retailer, keeps customers (it even gains trade-down shoppers) when the general price level goes up.
- **Unemployment:** each +1 pp of unemployment ≈ 100·β % lower weekly sales → sales are sensitive to the local labour market; plan inventory/promotions around regional economic conditions.

---
## 3. Model Diagnostics

### Task 3.1: Multicollinearity — Variance Inflation Factors

Compute VIF for each predictor in your final model. A VIF > 10 indicates problematic multicollinearity.

**Hint:** Use `variance_inflation_factor` from `statsmodels.stats.outliers_influence`.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF on the raw predictors (no constant), as used in the lecture
print(pd.Series([variance_inflation_factor(data[X].values, i) for i in range(len(X))], index=X, name='VIF').round(2))

### Task 3.2: Residual diagnostics

Create two plots:
1. **Fitted values vs residuals** — look for patterns (should be random scatter)
2. **Histogram of residuals** — check for normality

Also report the Durbin-Watson statistic from the model.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(model.fittedvalues, model.resid, s=5, alpha=.4); ax[0].axhline(0, c='r'); ax[0].set(xlabel='Fitted', ylabel='Residuals', title='Fitted vs residuals')
ax[1].hist(model.resid, bins=50); ax[1].set(title='Residual distribution'); plt.show()
print(f'Durbin-Watson: {sm.stats.durbin_watson(model.resid):.3f}')

### Discussion

**Answer the following:**
- Are the VIF values concerning? What causes multicollinearity in this dataset?
- What pattern do you see in the fitted-vs-residuals plot? What does it suggest?
- What does the Durbin-Watson statistic tell you? Why is autocorrelation expected in this data?

- **VIF:** high (>10) — computed without a constant, so it partly reflects that all predictors have large non-zero means; also CPI, unemployment and temperature co-move by region and time.
- **Fitted vs residuals:** clear **horizontal bands/stripes**, not random scatter → omitted store effects (each store's baseline level).
- **Durbin-Watson ≈ 0** → strong positive autocorrelation. Expected: the data is a panel of weekly time series, and each store's sales are persistent week to week (plus seasonality).

---
## 4. Store Fixed Effects

The low R-squared and extreme Durbin-Watson are caused by **unobserved store heterogeneity** — each store has a different baseline sales level driven by size, location, foot traffic, etc. By adding a dummy variable for each store, we absorb these differences and isolate the **within-store** variation in sales.

### Task 4.1: Fit the model with store fixed effects

Add `C(Store)` to the regression formula using `statsmodels.formula.api`. This creates 44 dummy variables (one per store, with one as reference).

**Hint:** Use `smf.ols()` with a formula string like `'y ~ x1 + x2 + C(Store)'`. Print only the key coefficients (not the 44 store dummies).

In [ ]:
import statsmodels.formula.api as smf

fe = smf.ols('log_Weekly_Sales ~ log_CPI + Temperature + Unemployment + Holiday_Flag + C(Store)', data).fit()
print(f'R2 = {fe.rsquared:.4f} | Adj. R2 = {fe.rsquared_adj:.4f}')
print(fe.summary2().tables[1].loc[['Intercept', 'log_CPI', 'Temperature', 'Unemployment', 'Holiday_Flag']])

### Task 4.2: Compare diagnostics — pooled vs fixed effects

Create a 2x2 subplot comparing residual plots from the pooled model (Section 2) and the fixed effects model:
- **Top row:** Pooled model — fitted vs residuals + histogram
- **Bottom row:** FE model — fitted vs residuals + histogram

Include R-squared and Durbin-Watson in each subplot title.

**Hint:** Use `durbin_watson()` from `statsmodels.stats.stattools`.

In [ ]:
from statsmodels.stats.stattools import durbin_watson

fig, ax = plt.subplots(2, 2, figsize=(14, 10))
for r, (n, m) in enumerate([('Pooled', model), ('Store FE', fe)]):
    ax[r, 0].scatter(m.fittedvalues, m.resid, s=5, alpha=.4); ax[r, 0].axhline(0, c='r'); ax[r, 0].set(xlabel='Fitted', ylabel='Residuals'); ax[r, 1].hist(m.resid, bins=50)
    for a in ax[r]: a.set_title(f'{n}: R²={m.rsquared:.3f}, DW={durbin_watson(m.resid):.2f}')
plt.tight_layout(); plt.show()

### Task 4.3: Single-store elasticity model

To build intuition for the within-store effect, zoom into **one specific store** and fit the same log-log model (`log_Weekly_Sales ~ log_CPI + Temperature + Unemployment`). This removes the panel structure entirely.

**Steps:**
1. Pick a store (e.g., the one with the highest sales variability)
2. Filter the data to that store only
3. Fit an OLS log-log model with `sm.add_constant()`
4. Print the summary and compare the CPI elasticity to the pooled and FE models

**Hint:** Use `data.groupby('Store')['log_Weekly_Sales'].std().idxmax()` to find the most variable store.

In [ ]:
s = data.groupby('Store')['log_Weekly_Sales'].std().idxmax(); d = data[data.Store == s]
ms = sm.OLS(d.log_Weekly_Sales, sm.add_constant(d[['log_CPI', 'Temperature', 'Unemployment']])).fit()
print(f'Store {s}'); print(ms.summary()); print(f"CPI elasticity (store {s}): {ms.params['log_CPI']:.3f}")
el = pd.Series({'Pooled': model.params['log_CPI'], f'Store {s}': ms.params['log_CPI'], 'Store FE': fe.params['log_CPI']})
el.plot.bar(rot=0, color=['grey', 'orange', 'steelblue'], title='CPI elasticity by approach'); plt.axhline(0, c='k'); plt.show()

### Discussion

**Answer the following:**
- How did R-squared and Durbin-Watson change with store fixed effects? Why?
- Did the residual banding disappear? What does this confirm?
- How does the single-store elasticity compare to the pooled and FE estimates? What does this tell you?
- Why is the fixed effects model more trustworthy than the pooled model for estimating elasticities?

- **R²** jumps to ~0.9+ and **DW** rises substantially: store dummies absorb the baseline level differences that dominated the residuals.
- **Banding disappears** → confirms the stripes were unobserved store heterogeneity.
- **Single store** elasticity differs from pooled and is closer in spirit to FE (within-store variation); differences show the pooled estimate is driven by **between-store** comparisons, not price changes over time.
- **FE is more trustworthy:** it compares each store with itself, removing bias from time-invariant confounders (size, location, demographics) correlated with regional CPI.

---
## 5. Interpretation and Business Implications

### Task 5.1: Summarise the elasticity estimates

Create a summary comparing the pooled model (Section 2) and the store fixed effects model (Section 4). Include:
- CPI elasticity estimate and interpretation
- Unemployment effect and interpretation
- R-squared and Durbin-Watson for both models
- Whether demand is elastic or inelastic

In [ ]:
res = [('Pooled', model), ('Store FE', fe)]
print(pd.DataFrame({n: {'CPI elasticity': r.params['log_CPI'], 'Unemployment': r.params['Unemployment'], 'R2': r.rsquared, 'DW': durbin_watson(r.resid)} for n, r in res}).round(4))
for n, r in res:
    e, u = r.params['log_CPI'], r.params['Unemployment']
    print(f"{n}: +1% CPI -> {e:+.2f}% sales ({'elastic' if abs(e) > 1 else 'inelastic'}); +1pp unemployment -> {100 * np.expm1(u):+.2f}% sales")

### Discussion

**Answer the following:**
- What is Walmart's estimated price elasticity? Is demand elastic or inelastic? Does this make business sense given Walmart's market positioning?
- Which macroeconomic factor has the largest practical impact on Walmart sales?
- What are the key limitations of this analysis? What data or methods would improve it?
- If you were advising Walmart's pricing team, what recommendations would you make based on these results?

- **Elasticity:** the FE `log_CPI` coefficient — |β| < 1 → **inelastic**. Consistent with Walmart's "Everyday Low Prices" positioning: necessity goods, customers trade down to Walmart when prices rise.
- **Largest practical impact:** **Unemployment** (local economic health) — a few pp swing moves sales noticeably.
- **Limitations:** CPI is a macro price index, not Walmart's own prices (no true own-price elasticity); omitted variables (promotions, competition, product mix); remaining autocorrelation/seasonality; possible endogeneity. Improvements: SKU-level price & quantity data, time fixed effects, clustered/HAC SEs, lagged terms or ARIMA errors, IV/experiments.
- **Recommendations:** moderate price increases in line with inflation are unlikely to cut revenue much; prioritise value messaging and inventory in high-unemployment regions; validate with store-level price experiments.

---
## 6. Regularised Regression: Ridge and Lasso

In Section 3 we found high VIF values (all > 10), meaning our predictors are highly multicollinear. When predictors are correlated, OLS coefficients become **unstable** — small changes in the data can produce large swings in the estimates.

**Regularisation** adds a penalty to the regression that shrinks coefficients toward zero:

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge** (L2) | $\lambda \sum \beta_j^2$ | Shrinks all coefficients — stabilises estimates but keeps all predictors |
| **Lasso** (L1) | $\lambda \sum |\beta_j|$ | Shrinks some coefficients to **exactly zero** — performs variable selection |

**Why does this matter for elasticity?** If multicollinearity inflates the CPI coefficient in the pooled model, regularisation will pull it toward a more stable (and possibly more accurate) estimate. If a predictor is truly irrelevant, Lasso will zero it out.

### Task 6.1: Fit Ridge and Lasso on the pooled model

Use cross-validation to find the optimal regularisation strength $\alpha$ for each method. Compare the coefficients to OLS.

**Steps:**
1. Standardise the predictors using `StandardScaler` (essential so penalties are comparable across features)
2. Fit `RidgeCV` and `LassoCV` from `sklearn.linear_model` with `cv=10`
3. Convert the standardised coefficients back to the original scale by dividing by `scaler.scale_`
4. Print a comparison table: OLS vs Ridge vs Lasso coefficients

**Hint:** Use `alphas = np.logspace(-3, 3, 100)` as the search grid.

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler

feats, alphas = ['log_CPI', 'Temperature', 'Unemployment'], np.logspace(-3, 3, 100)
scaler = StandardScaler(); Z = scaler.fit_transform(data[feats])
ridge, lasso = RidgeCV(alphas=alphas, cv=10).fit(Z, y), LassoCV(alphas=alphas, cv=10, max_iter=10000).fit(Z, y)
coefs = pd.DataFrame({'OLS': model.params.reindex(feats), 'Ridge': ridge.coef_ / scaler.scale_, 'Lasso': lasso.coef_ / scaler.scale_}, index=feats)
print(f'Best alpha: Ridge = {ridge.alpha_:.4g}, Lasso = {lasso.alpha_:.4g}'); print(coefs.round(5))

### Task 6.2: Visualise how regularisation shrinks coefficients

Plot the **coefficient paths** for Ridge and Lasso as regularisation strength increases. Use a 1x2 subplot grid (Ridge left, Lasso right).

**Steps:**
1. Loop over `alphas = np.logspace(-3, 3, 200)`, fitting Ridge and Lasso at each alpha
2. Store the standardised coefficients
3. Plot each predictor's coefficient as a line against alpha (log scale on x-axis)
4. Add a vertical line at the best alpha from cross-validation

**Hint:** Use `Ridge(alpha=a)` and `Lasso(alpha=a, max_iter=10000)` from `sklearn.linear_model`.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

A = np.logspace(-3, 3, 200)
paths = {n: np.array([M(alpha=a, **kw).fit(Z, y).coef_ for a in A]) for n, M, kw in [('Ridge', Ridge, {}), ('Lasso', Lasso, {'max_iter': 10000})]}
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, (n, P), best in zip(ax, paths.items(), [ridge.alpha_, lasso.alpha_]):
    for j, f in enumerate(feats): a.plot(A, P[:, j], label=f)
    a.axvline(best, ls='--', c='k', label='CV alpha'); a.set(xscale='log', xlabel='alpha', ylabel='Standardised coef', title=f'{n} coefficient path'); a.legend()
plt.tight_layout(); plt.show()

### Task 6.3: Compare OLS, Ridge, and Lasso elasticity estimates

Create a bar chart comparing the CPI elasticity across all methods: OLS (pooled), Ridge (pooled), Lasso (pooled), and OLS (store FE). Also check which predictors Lasso zeroed out.

In [ ]:
comp = pd.DataFrame({'OLS (pooled)': model.params, 'Ridge': coefs.Ridge, 'Lasso': coefs.Lasso, 'OLS (Store FE)': fe.params}).loc[feats]
comp.loc['log_CPI'].plot.bar(rot=0, title='CPI elasticity by method'); plt.axhline(0, c='k'); plt.show()
print('Unemployment coefficient:'); print(comp.loc['Unemployment'].round(5))
print('Lasso zeroed out:', coefs.index[coefs.Lasso == 0].tolist() or 'none')

### Discussion

**Answer the following:**
- How do the Ridge and Lasso CPI elasticity estimates compare to OLS? Are they larger, smaller, or similar?
- Did Lasso zero out any predictors? Does this agree with the backward selection results from Section 2?
- Looking at the coefficient path plots, which predictor is the most robust (last to be zeroed by Lasso)?
- Regularisation and fixed effects address different problems. What does each one fix? Could you combine them?
- If you had to advise Walmart's pricing team, which model's elasticity estimate would you trust most? Why?

- **Ridge/Lasso vs OLS:** similar, slightly shrunk toward zero; with n = 6,435 and only 3 predictors the CV-chosen penalty is small, so the change is minor.
- **Lasso zeroing:** at the CV alpha nothing (or very little) is zeroed — consistent with backward selection, which kept these predictors as significant.
- **Most robust predictor:** the last line to hit zero on the Lasso path (see plot) — the one with the largest standardised coefficient, typically `Unemployment`.
- **Different problems:** regularisation fixes **variance/instability from multicollinearity**; fixed effects fix **bias from omitted store heterogeneity**. Yes, combine them: Ridge/Lasso with store dummies (or on within-store demeaned data).
- **Most trusted:** the **store FE** estimate — regularisation cannot remove omitted-variable bias, while FE identifies the elasticity from within-store changes.